# Stage-1 Llama runner — Colab fallback

Vast.ai plus SSH/tmux is the primary execution path. This notebook remains a secondary interface and calls the same resumable Stage-1 runner. It does not run GPT-2 and does not include quarantined harmful-compliance data.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/ashioyajotham/safety_governor.git'
GIT_COMMIT = '<replace-with-immutable-commit>'
REPO_DIR = Path('/content/safety_governor')
CONFIG_PATH = 'configs/llama3_8b.yaml'
LAYERS = '0'
SITE = 'response_mean'
SPLIT = 'train'
RUN_ID = 'llama3-stage1-colab-layer0'
ARTIFACT_ROOT = Path('/content/drive/MyDrive/safety_governor_stage1')
HF_CACHE_ROOT = Path('/content/drive/MyDrive/safety_governor_hf_cache')
DTYPE = 'bfloat16'


In [ ]:
import os, subprocess, sys, yaml
from google.colab import drive, userdata

def run(command, cwd=None):
    print('$', ' '.join(map(str, command)))
    return subprocess.run(command, cwd=cwd, check=True, text=True)

if GIT_COMMIT.startswith('<'):
    raise ValueError('Set GIT_COMMIT to the immutable published commit first.')
drive.mount('/content/drive')
if REPO_DIR.exists():
    run(['git', 'fetch', 'origin'], cwd=REPO_DIR)
else:
    run(['git', 'clone', REPO_URL, str(REPO_DIR)])
run(['git', 'checkout', '--detach', GIT_COMMIT], cwd=REPO_DIR)
os.chdir(REPO_DIR)
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
token = userdata.get('HF_TOKEN')
if not token:
    raise ValueError('Add HF_TOKEN to Colab secrets.')
os.environ['HF_TOKEN'] = token
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
HF_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
profile_name = 'vast_bf16.yaml' if DTYPE == 'bfloat16' else 'vast_fp16.yaml'
profile = yaml.safe_load((Path('configs/runtime') / profile_name).read_text())
environment_lock = Path('/content/stage1-qualified-requirements.txt')
with environment_lock.open('w', encoding='utf-8') as handle:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], check=True, text=True, stdout=handle)
profile.update({'artifact_root': str(ARTIFACT_ROOT), 'hf_cache_root': str(HF_CACHE_ROOT), 'environment_lock': str(environment_lock), 'require_container_image': False})
runtime_profile = Path('/content/stage1_colab_runtime.yaml')
runtime_profile.write_text(yaml.safe_dump(profile), encoding='utf-8')


In [ ]:
run(['nvidia-smi'])
run([sys.executable, '-m', 'scripts.validate_dataset', 'datasets/frozen/english_contrastive.jsonl'])
run([sys.executable, '-m', 'scripts.stage1_preflight', CONFIG_PATH, '--runtime-profile', str(runtime_profile), '--check-model-access'])


In [ ]:
run([
    sys.executable, '-m', 'scripts.run_stage1', CONFIG_PATH,
    '--runtime-profile', str(runtime_profile),
    '--layers', LAYERS, '--site', SITE, '--split', SPLIT,
    '--run-id', RUN_ID, '--resume',
])


In [ ]:
import json
manifest = json.loads((ARTIFACT_ROOT / RUN_ID / 'manifest.json').read_text())
print(json.dumps({'run_id': manifest['run_id'], 'metrics': manifest['metrics'], 'config': manifest['config']}, indent=2))
for path in sorted((ARTIFACT_ROOT / RUN_ID).rglob('*')):
    if path.is_file(): print(path.relative_to(ARTIFACT_ROOT / RUN_ID), path.stat().st_size)
